# Drawing2CAD: SVG Vector (.npy) to SVG Converter

This notebook provides tools to convert Drawing2CAD's `svg_vec` format (.npy files) into standard SVG (Scalable Vector Graphics) files.

## Overview

**Drawing2CAD** uses a simplified vector engineering drawing representation focusing on core geometric information. This converter extracts the geometric data (LineTo and CubicBézier commands) and creates visual SVG files.

### Key Features

- **Accurate Implementation** based on the Drawing2CAD paper
- **Dual Command Support**: LineTo (L) and CubicBézier (C) commands
- **4-View Conversion**: Front, Top, Right, and Isometric views
- **Batch Processing**: Convert multiple files at once
- **Analysis Tools**: Inspect command types and statistics

### Format Specification (from the paper)

Each command uses **8 parameters**: X = (x₁, y₁, cx₁, cy₁, cx₂, cy₂, x₂, y₂)
- **LineTo (L)**: Straight lines with control points set to -1
- **CubicBézier (C)**: Curves using all 8 parameters
- **4 Views**: Front, Top, Right, Isometric (100 commands each)
- **Quantized**: Coordinates in range [0, 255]

### Quick Start

```python
# Convert a single file
npy_to_svg_all_views_improved('path/to/file.npy', 'output/dir')

# Batch convert multiple files
batch_convert_npy_to_svg('input/dir', 'output/dir', limit=10)
```

---

In [20]:
import numpy as np

In [21]:
x=np.load(r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec\0000\00000134.npy')

In [22]:
print(x.shape)

(400, 10)


In [23]:
# Examine the structure
print("First 10 rows:")
print(x[:10])
print("\nData range:")
print(f"Min: {x.min()}, Max: {x.max()}")
print("\nFirst column (view indices):")
print(np.unique(x[:, 0], return_counts=True))

First 10 rows:
[[  0.   0.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   2. 156.  63.  -1.  -1.  -1.  -1.  98.  63.]
 [  0.   2.  98.  63.  -1.  -1.  -1.  -1.  98. 191.]
 [  0.   2.  98. 191.  -1.  -1.  -1.  -1. 156. 191.]
 [  0.   2. 156. 191.  -1.  -1.  -1.  -1. 156.  63.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]]

Data range:
Min: -1.0, Max: 215.0

First column (view indices):
(array([0., 1., 2., 3.], dtype=float32), array([100, 100, 100, 100], dtype=int64))


In [24]:
import svgwrite

def npy_to_svg(npy_data, output_path, view_idx=0, canvas_size=256):
    """
    Convert a single view from .npy data to SVG file.
    
    Args:
        npy_data: (400, 10) array containing 4 views
        output_path: Output SVG file path
        view_idx: Which view to convert (0=Front, 1=Top, 2=Right, 3=Iso)
        canvas_size: SVG canvas size in pixels
    """
    # Extract the specific view (100 rows per view)
    view_start = view_idx * 100
    view_end = view_start + 100
    view_data = npy_data[view_start:view_end]
    
    # Create SVG drawing
    dwg = svgwrite.Drawing(output_path, size=(canvas_size, canvas_size), 
                           viewBox=f'0 0 {canvas_size} {canvas_size}')
    
    # Add white background
    dwg.add(dwg.rect(insert=(0, 0), size=('100%', '100%'), fill='white'))
    
    # Process each row in the view
    for row in view_data:
        view_id = int(row[0])
        cmd_type = int(row[1])
        
        # Skip padding rows
        if cmd_type < 0:
            continue
        
        # Extract coordinate data (columns 2-9)
        coords = row[2:]
        
        # Command type interpretation:
        # 0 = moveto/start path
        # 1 = padding/end marker
        # 2 = lineto (line segment)
        # Other types may include arc, circle, etc.
        
        if cmd_type == 2:  # Line segment
            # Coordinates: [x1, y1, ?, ?, ?, ?, x2, y2]
            x1, y1 = float(coords[0]), float(coords[1])
            x2, y2 = float(coords[6]), float(coords[7])
            
            # Skip invalid coordinates
            if x1 < 0 or y1 < 0 or x2 < 0 or y2 < 0:
                continue
            
            # Draw line
            dwg.add(dwg.line(start=(x1, y1), end=(x2, y2), 
                           stroke='black', stroke_width=1))
        
        elif cmd_type == 3:  # Arc (hypothetical)
            # You would need to implement arc drawing based on the actual format
            pass
        
        elif cmd_type == 4:  # Circle (hypothetical)
            # Extract center and radius
            cx, cy = float(coords[0]), float(coords[1])
            r = float(coords[4]) if len(coords) > 4 and coords[4] >= 0 else 10
            
            if cx >= 0 and cy >= 0 and r > 0:
                dwg.add(dwg.circle(center=(cx, cy), r=r, 
                               stroke='black', fill='none', stroke_width=1))
    
    dwg.save()
    print(f"Saved SVG to {output_path}")


def npy_to_svg_all_views(npy_path, output_dir):
    """
    Convert all 4 views from a .npy file to separate SVG files.
    
    Args:
        npy_path: Path to input .npy file
        output_dir: Directory to save output SVG files
    """
    import os
    
    # Load data
    data = np.load(npy_path)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get base filename
    base_name = os.path.splitext(os.path.basename(npy_path))[0]
    
    view_names = ['front', 'top', 'right', 'isometric']
    
    # Convert each view
    for view_idx, view_name in enumerate(view_names):
        output_path = os.path.join(output_dir, f"{base_name}_{view_name}.svg")
        npy_to_svg(data, output_path, view_idx=view_idx)
    
    print(f"All views saved to {output_dir}")

In [29]:
# Test the converter with the loaded data
output_dir = r'C:\Users\LEGION\Desktop\cad_project\svg_output'
npy_to_svg_all_views(
    r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec_original\0000\00000007.npy',
    output_dir
)

Saved SVG to C:\Users\LEGION\Desktop\cad_project\svg_output\00000007_front.svg
Saved SVG to C:\Users\LEGION\Desktop\cad_project\svg_output\00000007_top.svg
Saved SVG to C:\Users\LEGION\Desktop\cad_project\svg_output\00000007_right.svg
Saved SVG to C:\Users\LEGION\Desktop\cad_project\svg_output\00000007_isometric.svg
All views saved to C:\Users\LEGION\Desktop\cad_project\svg_output


In [12]:
# Analyze the command types in the data
unique_cmds = np.unique(x[:, 1])
print("Unique command types:", unique_cmds)

# Count each command type per view
for view_idx in range(4):
    view_data = x[view_idx*100:(view_idx+1)*100]
    cmd_counts = {}
    for cmd in unique_cmds:
        count = np.sum(view_data[:, 1] == cmd)
        if count > 0:
            cmd_counts[int(cmd)] = count
    print(f"\nView {view_idx} command distribution:", cmd_counts)

Unique command types: [0. 1. 2.]

View 0 command distribution: {0: 1, 1: 95, 2: 4}

View 1 command distribution: {0: 1, 1: 95, 2: 4}

View 2 command distribution: {0: 1, 1: 95, 2: 4}

View 3 command distribution: {0: 1, 1: 90, 2: 9}


In [13]:
# Look at non-padding rows from view 0
view0 = x[0:100]
non_padding = view0[view0[:, 1] != 1]
print("Non-padding rows from view 0:")
print(non_padding)

Non-padding rows from view 0:
[[  0.   0.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   2. 156.  63.  -1.  -1.  -1.  -1.  98.  63.]
 [  0.   2.  98.  63.  -1.  -1.  -1.  -1.  98. 191.]
 [  0.   2.  98. 191.  -1.  -1.  -1.  -1. 156. 191.]
 [  0.   2. 156. 191.  -1.  -1.  -1.  -1. 156.  63.]]


In [39]:
import svgwrite
import os

def npy_to_svg_improved(npy_data, output_path, view_idx=0, canvas_size=256):
    """
    Convert a single view from .npy data to SVG file with proper path handling.
    
    The .npy format for svg_vec in Drawing2CAD (from the paper):
    - Shape: (400, 10) containing 4 views of 100 command sequences each
    - Column 0: View label (0=Front, 1=Top, 2=Right, 3=Isometric)
    - Column 1: Command type
        * 0: <SOS> (Start of sequence)
        * 1: Padding/empty
        * 2: L (LineTo) - straight line
        * 3: C (CubicBézier) - Bézier curve
        * 4: <EOS> (End of sequence)
    - Columns 2-9: 8-dimensional command parameters X = (x₁, y₁, cx₁, cy₁, cx₂, cy₂, x₂, y₂)
        * For LineTo: [x₁, y₁, -1, -1, -1, -1, x₂, y₂]
        * For CubicBézier: [x₁, y₁, cx₁, cy₁, cx₂, cy₂, x₂, y₂] (all 8 parameters)
    
    Args:
        npy_data: (400, 10) array containing 4 views
        output_path: Output SVG file path
        view_idx: Which view to convert (0=Front, 1=Top, 2=Right, 3=Isometric)
        canvas_size: SVG canvas size in pixels
    """
    # Extract the specific view (100 command sequences per view)
    view_start = view_idx * 100
    view_end = view_start + 100
    view_data = npy_data[view_start:view_end]
    
    # Create SVG drawing
    dwg = svgwrite.Drawing(output_path, size=(canvas_size, canvas_size), 
                           viewBox=f'0 0 {canvas_size} {canvas_size}')
    
    # Add white background
    dwg.add(dwg.rect(insert=(0, 0), size=('100%', '100%'), fill='white'))
    
    # Collect all drawing commands
    line_segments = []
    bezier_curves = []
    
    for row in view_data:
        cmd_type = int(row[1])
        
        # Skip padding, SOS, and EOS markers
        if cmd_type == 0 or cmd_type == 1 or cmd_type == 4:
            continue
        
        # Extract 8-parameter coordinates
        params = row[2:]
        x1, y1 = float(params[0]), float(params[1])
        cx1, cy1 = float(params[2]), float(params[3])
        cx2, cy2 = float(params[4]), float(params[5])
        x2, y2 = float(params[6]), float(params[7])
        
        # Skip invalid start/end coordinates
        if x1 < 0 or y1 < 0 or x2 < 0 or y2 < 0:
            continue
        
        if cmd_type == 2:  # LineTo (L) command
            line_segments.append(((x1, y1), (x2, y2)))
            
        elif cmd_type == 3:  # CubicBézier (C) command
            # Check if control points are valid
            if cx1 >= 0 and cy1 >= 0 and cx2 >= 0 and cy2 >= 0:
                bezier_curves.append(((x1, y1), (cx1, cy1), (cx2, cy2), (x2, y2)))
            else:
                # If control points invalid, treat as line
                line_segments.append(((x1, y1), (x2, y2)))
    
    # Draw line segments
    for (x1, y1), (x2, y2) in line_segments:
        dwg.add(dwg.line(start=(x1, y1), end=(x2, y2), 
                       stroke='black', stroke_width=0.1, 
                       stroke_linecap='round'))
    
    # Draw Bézier curves
    for (x1, y1), (cx1, cy1), (cx2, cy2), (x2, y2) in bezier_curves:
        # SVG path for cubic Bézier: M x1,y1 C cx1,cy1 cx2,cy2 x2,y2
        path_data = f"M {x1},{y1} C {cx1},{cy1} {cx2},{cy2} {x2},{y2}"
        dwg.add(dwg.path(d=path_data, fill='none', stroke='blue', 
                        stroke_width=0.1, stroke_linecap='round'))
    
    dwg.save()
    return len(line_segments), len(bezier_curves)


def npy_to_svg_all_views_improved(npy_path, output_dir):
    """
    Convert all 4 views from a .npy file to separate SVG files.
    
    Args:
        npy_path: Path to input .npy file
        output_dir: Directory to save output SVG files
    """
    # Load data
    data = np.load(npy_path)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get base filename
    base_name = os.path.splitext(os.path.basename(npy_path))[0]
    
    view_names = ['front', 'top', 'right', 'isometric']
    
    print(f"Converting {base_name}:")
    
    # Convert each view
    total_lines = 0
    total_curves = 0
    for view_idx, view_name in enumerate(view_names):
        output_path = os.path.join(output_dir, f"{base_name}_{view_name}.svg")
        num_lines, num_curves = npy_to_svg_improved(data, output_path, view_idx=view_idx)
        total_lines += num_lines
        total_curves += num_curves
        print(f"  {view_name}: {num_lines} lines, {num_curves} Bézier curves -> {output_path}")
    
    print(f"✓ All views saved to {output_dir}")
    print(f"  Total: {total_lines} lines, {total_curves} Bézier curves")


def batch_convert_npy_to_svg(input_dir, output_dir, limit=None):
    """
    Batch convert all .npy files in a directory to SVG.
    
    Args:
        input_dir: Directory containing .npy files (can have subdirectories)
        output_dir: Base directory for output SVG files
        limit: Maximum number of files to process (None = all)
    """
    import glob
    
    # Find all .npy files recursively
    npy_files = []
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith('.npy'):
                npy_files.append(os.path.join(root, file))
    
    if limit:
        npy_files = npy_files[:limit]
    
    print(f"Found {len(npy_files)} .npy files")
    
    for i, npy_path in enumerate(npy_files, 1):
        try:
            # Get relative path structure
            rel_path = os.path.relpath(os.path.dirname(npy_path), input_dir)
            out_subdir = os.path.join(output_dir, rel_path)
            
            print(f"\n[{i}/{len(npy_files)}] {os.path.basename(npy_path)}")
            npy_to_svg_all_views_improved(npy_path, out_subdir)
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\n{'='*60}")
    print(f"Batch conversion complete: {len(npy_files)} files processed")
    print(f"Output directory: {output_dir}")

In [43]:
# Test the improved converter
output_dir_improved = r'C:\Users\LEGION\Desktop\cad_project\svg_output_improved'
npy_to_svg_all_views_improved(
    r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec_original\0001\00017143.npy',
    output_dir_improved
)

Converting 00017143:
  front: 13 lines, 4 Bézier curves -> C:\Users\LEGION\Desktop\cad_project\svg_output_improved\00017143_front.svg
  top: 10 lines, 0 Bézier curves -> C:\Users\LEGION\Desktop\cad_project\svg_output_improved\00017143_top.svg
  right: 9 lines, 0 Bézier curves -> C:\Users\LEGION\Desktop\cad_project\svg_output_improved\00017143_right.svg
  isometric: 24 lines, 3 Bézier curves -> C:\Users\LEGION\Desktop\cad_project\svg_output_improved\00017143_isometric.svg
✓ All views saved to C:\Users\LEGION\Desktop\cad_project\svg_output_improved
  Total: 56 lines, 7 Bézier curves


## NPY to SVG Converter for Drawing2CAD

This notebook converts `.npy` files from the Drawing2CAD `svg_vec` format to standard `.svg` (Scalable Vector Graphics) files.

### Understanding the svg_vec .npy Format

Based on the paper **"Drawing2CAD: Sequence-to-Sequence Learning for CAD Generation from Vector Drawings"**:

**Representation:**
- Each vector engineering drawing uses a simplified representation focusing on core geometric information
- Command set includes: `<SOS>`, `L` (LineTo), `C` (CubicBézier), `<EOS>`
- Each command has an 8-value parameter list: **X = (x₁, y₁, cx₁, cy₁, cx₂, cy₂, x₂, y₂) ∈ ℝ⁸**
  - x₁, y₁: Start point
  - x₂, y₂: End point
  - cx₁, cy₁, cx₂, cy₂: Control points for Bézier curves
  - For LineTo commands: only start/end points used, control points set to **-1**
  - For CubicBézier commands: all 8 parameters are used

**Data Structure:**
- **Shape**: `(400, 10)` - 4 engineering drawing views × 100 command sequences each
- Each CAD model corresponds to 4 views: **Front, Top, Right, Isometric**

**Views (Row Ranges)**:
- **Front view**: Rows 0-99 (view_idx=0)
- **Top view**: Rows 100-199 (view_idx=1)  
- **Right view**: Rows 200-299 (view_idx=2)
- **Isometric view**: Rows 300-399 (view_idx=3)

**Column Schema:**

| Column | Description | Values |
|--------|-------------|--------|
| 0 | View label | 0=Front, 1=Top, 2=Right, 3=Isometric |
| 1 | Command type | 0=`<SOS>`, 1=Padding, 2=`L` (LineTo), 3=`C` (CubicBézier), 4=`<EOS>` |
| 2-3 | Start point (x₁, y₁) | 0-255 (quantized) or -1 (unused) |
| 4-5 | Control point 1 (cx₁, cy₁) | 0-255 or **-1 for LineTo** |
| 6-7 | Control point 2 (cx₂, cy₂) | 0-255 or **-1 for LineTo** |
| 8-9 | End point (x₂, y₂) | 0-255 (quantized) or -1 (unused) |

**Command Types**:
- **0 (`<SOS>`)**: Start of sequence marker
- **1**: Padding/empty row (unused)
- **2 (`L`)**: LineTo command - straight line from (x₁,y₁) to (x₂,y₂), control points are -1
- **3 (`C`)**: CubicBézier command - curve using all 8 parameters
- **4 (`<EOS>`)**: End of sequence marker

### Usage Examples

See cells below for:
1. Single file conversion
2. Batch conversion of multiple files
3. Custom processing with Bézier curve support

### Example: Batch Convert Multiple Files

In [ ]:
# Example: Convert first 5 files from the svg_vec directory
# Uncomment to run:
# batch_convert_npy_to_svg(
#     input_dir=r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec',
#     output_dir=r'C:\Users\LEGION\Desktop\cad_project\svg_batch_output',
#     limit=5  # Remove limit to convert all files
# )

### Advanced: Create Closed Paths

For better visualization, we can detect and create closed paths:

In [16]:
def npy_to_svg_with_paths(npy_data, output_path, view_idx=0, canvas_size=256):
    """
    Convert .npy to SVG with closed paths for better rendering.
    Attempts to connect line segments into continuous paths.
    """
    view_start = view_idx * 100
    view_end = view_start + 100
    view_data = npy_data[view_start:view_end]
    
    # Create SVG drawing
    dwg = svgwrite.Drawing(output_path, size=(canvas_size, canvas_size), 
                           viewBox=f'0 0 {canvas_size} {canvas_size}')
    
    # Add white background
    dwg.add(dwg.rect(insert=(0, 0), size=('100%', '100%'), fill='white'))
    
    # Collect all line segments
    segments = []
    for row in view_data:
        cmd_type = int(row[1])
        if cmd_type != 2:
            continue
        
        coords = row[2:]
        x1, y1 = float(coords[0]), float(coords[1])
        x2, y2 = float(coords[6]), float(coords[7])
        
        if x1 >= 0 and y1 >= 0 and x2 >= 0 and y2 >= 0:
            segments.append(((x1, y1), (x2, y2)))
    
    if not segments:
        dwg.save()
        return 0
    
    # Create a path connecting all segments
    path_commands = []
    
    # Start with first segment
    path_commands.append(f"M {segments[0][0][0]} {segments[0][0][1]}")
    path_commands.append(f"L {segments[0][1][0]} {segments[0][1][1]}")
    
    # Add remaining segments
    for (x1, y1), (x2, y2) in segments[1:]:
        # Check if this segment connects to the previous endpoint
        prev_end = (float(path_commands[-1].split()[1]), float(path_commands[-1].split()[2]))
        
        if abs(prev_end[0] - x1) < 0.1 and abs(prev_end[1] - y1) < 0.1:
            # Continues from previous point
            path_commands.append(f"L {x2} {y2}")
        else:
            # Starts a new segment
            path_commands.append(f"M {x1} {y1}")
            path_commands.append(f"L {x2} {y2}")
    
    # Try to close the path if it forms a loop
    first_pt = segments[0][0]
    last_pt = (float(path_commands[-1].split()[1]), float(path_commands[-1].split()[2]))
    
    if abs(first_pt[0] - last_pt[0]) < 0.1 and abs(first_pt[1] - last_pt[1]) < 0.1:
        path_commands.append("Z")
    
    # Create and add the path
    path_str = " ".join(path_commands)
    dwg.add(dwg.path(d=path_str, fill='lightgray', stroke='black', 
                    stroke_width=2, stroke_linejoin='round'))
    
    dwg.save()
    return len(segments)


# Test with paths
output_path_version = r'C:\Users\LEGION\Desktop\cad_project\svg_output_paths'
os.makedirs(output_path_version, exist_ok=True)

view_names = ['front', 'top', 'right', 'isometric']
print("Creating SVGs with closed paths:")
for view_idx, view_name in enumerate(view_names):
    output = os.path.join(output_path_version, f"00000134_{view_name}_path.svg")
    num_seg = npy_to_svg_with_paths(x, output, view_idx=view_idx)
    print(f"  {view_name}: {num_seg} segments")

Creating SVGs with closed paths:
  front: 4 segments
  top: 4 segments
  right: 4 segments
  isometric: 9 segments


### Summary and Verification

The converters are now ready to use! You can:

1. **View the generated SVG files** by opening them in:
   - Any web browser (Chrome, Firefox, Edge, etc.)
   - Vector graphics editors (Inkscape, Adobe Illustrator, etc.)
   - VS Code (with an SVG viewer extension)

2. **Understand the data** by examining the analysis cells above

3. **Customize** the converters for your specific needs (e.g., different colors, stroke widths, fill styles)

### Files Created

Check the following directories for outputs:
- `svg_output_improved/` - Individual line segments
- `svg_output_paths/` - Closed paths with fill

In [17]:
# Verify output files were created
import glob

print("="*60)
print("VERIFICATION: Checking created SVG files")
print("="*60)

output_dirs = [
    r'C:\Users\LEGION\Desktop\cad_project\svg_output_improved',
    r'C:\Users\LEGION\Desktop\cad_project\svg_output_paths'
]

for output_dir in output_dirs:
    if os.path.exists(output_dir):
        svg_files = glob.glob(os.path.join(output_dir, '*.svg'))
        print(f"\n{os.path.basename(output_dir)}:")
        print(f"  Found {len(svg_files)} SVG files")
        for f in svg_files[:5]:  # Show first 5
            file_size = os.path.getsize(f) / 1024  # KB
            print(f"    ✓ {os.path.basename(f)} ({file_size:.2f} KB)")
        if len(svg_files) > 5:
            print(f"    ... and {len(svg_files) - 5} more")
    else:
        print(f"\n{output_dir}: Not found")

print("\n" + "="*60)
print("You can open these .svg files in your web browser to view them!")
print("="*60)

VERIFICATION: Checking created SVG files

svg_output_improved:
  Found 4 SVG files
    ✓ 00000134_front.svg (0.73 KB)
    ✓ 00000134_isometric.svg (1.25 KB)
    ✓ 00000134_right.svg (0.73 KB)
    ✓ 00000134_top.svg (0.73 KB)

svg_output_paths:
  Found 4 SVG files
    ✓ 00000134_front_path.svg (0.46 KB)
    ✓ 00000134_isometric_path.svg (0.57 KB)
    ✓ 00000134_right_path.svg (0.46 KB)
    ✓ 00000134_top_path.svg (0.46 KB)

You can open these .svg files in your web browser to view them!


---

## Technical Details: NPY Format Specification

Based on the paper **"Drawing2CAD: Sequence-to-Sequence Learning for CAD Generation from Vector Drawings"**:

### Vector Engineering Drawings Representation

The representation focuses exclusively on core geometric information, excluding non-essential path attributes (visibility, color, fill properties).

**Command Set**: `{<SOS>, L, C, <EOS>}`
- **L**: LineTo command (straight line)
- **C**: CubicBézier command (Bézier curve)
- **<SOS>**: Start of sequence marker
- **<EOS>**: End of sequence marker

### Standardized 8-Parameter Format

Each command uses an 8-value parameter list:

**X = (x₁, y₁, cx₁, cy₁, cx₂, cy₂, x₂, y₂) ∈ ℝ⁸**

Where:
- **(x₁, y₁)**: Start point
- **(x₂, y₂)**: End point  
- **(cx₁, cy₁)**: First control point (for Bézier curves)
- **(cx₂, cy₂)**: Second control point (for Bézier curves)

**Command-Specific Usage**:
- **LineTo (L)**: Only start and end points are utilized; control point parameters are set to **-1**
- **CubicBézier (C)**: All eight parameters are employed

### Data Format in .npy Files

**Array Shape**: `(400, 10)`
- 4 views × 100 command sequences per view
- 10 columns per row

**Structure**: Each CAD model corresponds to four engineering drawings

**Views (Row Ranges)**:
- **Front view**: Rows 0-99 (view_idx=0)
- **Top view**: Rows 100-199 (view_idx=1)  
- **Right view**: Rows 200-299 (view_idx=2)
- **Isometric view**: Rows 300-399 (view_idx=3)

**Column Schema**:

| Column | Parameter | Description |
|--------|-----------|-------------|
| 0 | View Label vᵢ | 0=Front, 1=Top, 2=Right, 3=Isometric |
| 1 | Command Type cⱼᵢ | 0=`<SOS>`, 1=Padding, 2=`L`, 3=`C`, 4=`<EOS>` |
| 2 | x₁ | Start point x-coordinate (quantized 0-255) |
| 3 | y₁ | Start point y-coordinate (quantized 0-255) |
| 4 | cx₁ | Control point 1 x-coordinate (or -1 for LineTo) |
| 5 | cy₁ | Control point 1 y-coordinate (or -1 for LineTo) |
| 6 | cx₂ | Control point 2 x-coordinate (or -1 for LineTo) |
| 7 | cy₂ | Control point 2 y-coordinate (or -1 for LineTo) |
| 8 | x₂ | End point x-coordinate (quantized 0-255) |
| 9 | y₂ | End point y-coordinate (quantized 0-255) |

### Coordinate System

- Coordinates are quantized to the range **[0, 255]**
- Origin is at top-left (standard SVG convention)
- Canvas size is **256×256** pixels

### Mathematical Formulation

Each vector engineering drawing is defined as an ordered sequence:

**Dᵢ = {S₁, S₂, ..., Sₙ}**

where:
- **Dᵢ**: The i-th engineering drawing
- **N = 100**: Number of command sequences
- **Sᵢ = (vᵢ, Cᵢ)**: Sub-sequence with view label and command
- **Cᵢ = (cⱼᵢ, Xⱼᵢ)**: Command type and 8D parameters

---

## Additional Utilities

In [30]:
def analyze_npy_file(npy_path):
    """
    Analyze a .npy file and print statistics about its content.
    
    Args:
        npy_path: Path to .npy file
    """
    data = np.load(npy_path)
    
    print(f"File: {os.path.basename(npy_path)}")
    print(f"Shape: {data.shape}")
    print(f"Data range: [{data.min():.1f}, {data.max():.1f}]")
    print(f"Data type: {data.dtype}")
    
    # Analyze each view
    view_names = ['Front', 'Top', 'Right', 'Isometric']
    print(f"\n{'View':<12} {'<SOS>':<8} {'LineTo(L)':<12} {'Bézier(C)':<12} {'<EOS>':<8} {'Padding':<8}")
    print("-" * 65)
    
    for view_idx, view_name in enumerate(view_names):
        view_data = data[view_idx*100:(view_idx+1)*100]
        
        # Count command types according to the paper
        cmd_col = view_data[:, 1]
        sos_count = np.sum(cmd_col == 0)     # <SOS>
        line_count = np.sum(cmd_col == 2)    # L (LineTo)
        bezier_count = np.sum(cmd_col == 3)  # C (CubicBézier)
        eos_count = np.sum(cmd_col == 4)     # <EOS>
        padding_count = np.sum(cmd_col == 1) # Padding
        
        print(f"{view_name:<12} {sos_count:<8} {line_count:<12} {bezier_count:<12} {eos_count:<8} {padding_count:<8}")
    
    print()
    
    # Additional statistics
    print("Parameter Statistics:")
    print(f"  Valid coordinates (>= 0): {np.sum(data[:, 2:] >= 0)}")
    print(f"  Control points set to -1 (LineTo): {np.sum(data[:, 4:8] == -1)}")
    

def compare_npy_files(npy_path1, npy_path2):
    """
    Compare two .npy files to see if they have similar structures.
    
    Args:
        npy_path1, npy_path2: Paths to .npy files to compare
    """
    data1 = np.load(npy_path1)
    data2 = np.load(npy_path2)
    
    print(f"Comparing:")
    print(f"  File 1: {os.path.basename(npy_path1)}")
    print(f"  File 2: {os.path.basename(npy_path2)}")
    print()
    
    print(f"Shape match: {data1.shape == data2.shape}")
    print(f"Identical: {np.allclose(data1, data2)}")
    print(f"Max difference: {np.abs(data1 - data2).max():.2f}")
    print()


# Example usage
print("="*65)
print("ANALYZING NPY FILE - Drawing2CAD Format")
print("="*65)
analyze_npy_file(r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec\0000\00000134.npy')

ANALYZING NPY FILE - Drawing2CAD Format
File: 00000134.npy
Shape: (400, 10)
Data range: [-1.0, 215.0]
Data type: float32

View         <SOS>    LineTo(L)    Bézier(C)    <EOS>    Padding 
-----------------------------------------------------------------
Front        1        4            0            0        95      
Top          1        4            0            0        95      
Right        1        4            0            0        95      
Isometric    1        9            0            0        90      

Parameter Statistics:
  Valid coordinates (>= 0): 84
  Control points set to -1 (LineTo): 1600


---

## Complete Workflow Example

Here's a complete workflow for converting NPY files to SVG:

In [19]:
"""
COMPLETE WORKFLOW EXAMPLE
==========================

Step 1: Analyze the file first
Step 2: Convert to SVG
Step 3: Verify the output

Uncomment and run to use:
"""

# # Step 1: Analyze
# input_file = r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec\0000\00000134.npy'
# analyze_npy_file(input_file)

# # Step 2: Convert (choose one method)

# # Option A: Individual line segments
# output_dir = r'C:\Users\LEGION\Desktop\cad_project\my_svg_output'
# npy_to_svg_all_views_improved(input_file, output_dir)

# # Option B: Closed paths with fill
# output_dir_paths = r'C:\Users\LEGION\Desktop\cad_project\my_svg_output_paths'
# os.makedirs(output_dir_paths, exist_ok=True)
# for view_idx, view_name in enumerate(['front', 'top', 'right', 'isometric']):
#     data = np.load(input_file)
#     output = os.path.join(output_dir_paths, f"{view_name}.svg")
#     npy_to_svg_with_paths(data, output, view_idx=view_idx)

# # Step 3: Batch convert multiple files
# batch_convert_npy_to_svg(
#     input_dir=r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec',
#     output_dir=r'C:\Users\LEGION\Desktop\cad_project\batch_svg_output',
#     limit=10  # Convert first 10 files, or remove to convert all
# )

print("Workflow template ready! Uncomment the code above to run.")

Workflow template ready! Uncomment the code above to run.


In [ ]:
## Batch Convert Converted_PNGs Samples

# Convert NPY files corresponding to PNG files in the Converted_PNGs folder:

In [44]:
import os
import glob
from pathlib import Path

def convert_converted_pngs_samples():
    """
    Convert NPY files corresponding to PNG samples in Converted_PNGs folder.
    
    Process:
    1. Find all PNGs in Converted_PNGs folder
    2. For each PNG, locate corresponding .npy in svg_vec_original
    3. Convert to SVG and save in Npy_SVGs folder
    """
    
    # Define paths
    png_dir = r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Converted_PNGs'
    npy_base_dir = r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec_original'
    output_dir = r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs'
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Find all PNG files
    png_files = glob.glob(os.path.join(png_dir, '*.png'))
    
    if not png_files:
        print(f"No PNG files found in {png_dir}")
        return
    
    print(f"Found {len(png_files)} PNG files in Converted_PNGs")
    print(f"="*70)
    
    successful = 0
    failed = 0
    
    for i, png_path in enumerate(png_files, 1):
        # Extract base name (e.g., 00021965 from 00021965.png)
        base_name = os.path.splitext(os.path.basename(png_path))[0]
        
        # Determine subdirectory (first 4 digits -> 0002 from 00021965)
        if len(base_name) >= 4:
            subdir = base_name[:4]
        else:
            print(f"[{i}/{len(png_files)}] ✗ Invalid filename format: {base_name}")
            failed += 1
            continue
        
        # Construct NPY path
        npy_path = os.path.join(npy_base_dir, subdir, f"{base_name}.npy")
        
        # Check if NPY exists
        if not os.path.exists(npy_path):
            print(f"[{i}/{len(png_files)}] ✗ NPY not found: {base_name}")
            print(f"  Expected: {npy_path}")
            failed += 1
            continue
        
        try:
            # Load NPY data
            data = np.load(npy_path)
            
            # Create subdirectory in output for this sample
            sample_output_dir = os.path.join(output_dir, base_name)
            os.makedirs(sample_output_dir, exist_ok=True)
            
            # Convert all 4 views
            view_names = ['front', 'top', 'right', 'isometric']
            total_lines = 0
            total_curves = 0
            
            for view_idx, view_name in enumerate(view_names):
                svg_output = os.path.join(sample_output_dir, f"{base_name}_{view_name}.svg")
                num_lines, num_curves = npy_to_svg_improved(data, svg_output, view_idx=view_idx)
                total_lines += num_lines
                total_curves += num_curves
            
            print(f"[{i}/{len(png_files)}] ✓ {base_name}: {total_lines} lines, {total_curves} curves -> {sample_output_dir}")
            successful += 1
            
        except Exception as e:
            print(f"[{i}/{len(png_files)}] ✗ Error converting {base_name}: {e}")
            failed += 1
    
    print(f"\n{'='*70}")
    print(f"Conversion Summary:")
    print(f"  Total PNG files: {len(png_files)}")
    print(f"  Successfully converted: {successful}")
    print(f"  Failed: {failed}")
    print(f"  Output directory: {output_dir}")
    print(f"{'='*70}")


# Run the conversion
convert_converted_pngs_samples()

Found 99 PNG files in Converted_PNGs
[1/99] ✓ 00011434: 41 lines, 10 curves -> C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs\00011434
[2/99] ✓ 00021965: 21 lines, 27 curves -> C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs\00021965
[3/99] ✓ 00022626: 21 lines, 27 curves -> C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs\00022626
[4/99] ✓ 00038386: 41 lines, 0 curves -> C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs\00038386
[5/99] ✓ 00044521: 12 lines, 12 curves -> C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs\00044521
[6/99] ✓ 00058885: 47 lines, 10 curves -> C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs\00058885
[7/99] ✓ 00059560: 18 lines, 6 curves -> C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs\00059560
[8/99] ✓ 00064136: 36 lines, 13 curves -> C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs\00064136
[9/99] ✓ 00080711: 21 lines, 0 curves -> C:\Users\LEGION\Desktop\cad_project\DeepCAD\

In [45]:
# Verify the conversion results
npy_svgs_dir = r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs'

if os.path.exists(npy_svgs_dir):
    # Count subdirectories (one per sample)
    sample_dirs = [d for d in os.listdir(npy_svgs_dir) 
                   if os.path.isdir(os.path.join(npy_svgs_dir, d))]
    
    print(f"✓ Created {len(sample_dirs)} sample directories in Npy_SVGs")
    
    # Show first few samples
    print(f"\nFirst 5 samples:")
    for sample_dir in sorted(sample_dirs)[:5]:
        sample_path = os.path.join(npy_svgs_dir, sample_dir)
        svg_files = glob.glob(os.path.join(sample_path, '*.svg'))
        print(f"  {sample_dir}: {len(svg_files)} SVG files")
    
    # Count total SVG files
    total_svgs = sum(1 for root, dirs, files in os.walk(npy_svgs_dir) 
                     for f in files if f.endswith('.svg'))
    print(f"\n✓ Total SVG files created: {total_svgs}")
    print(f"✓ Output directory: {npy_svgs_dir}")
else:
    print(f"✗ Directory not found: {npy_svgs_dir}")

✓ Created 99 sample directories in Npy_SVGs

First 5 samples:
  00011434: 4 SVG files
  00021965: 4 SVG files
  00022626: 4 SVG files
  00038386: 4 SVG files
  00044521: 4 SVG files

✓ Total SVG files created: 396
✓ Output directory: C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\Npy_SVGs
